## Setup

In [181]:
import math 
import random 
import numpy as np 
import pandas as pd 
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

## I. CONSTANT VARIABLES FOR SETUP

In [228]:
SEED = 42 
NUM_USERS = 1000 
NUM_ITEMS = 500 
NUM_CATEGORIES = 10 
MIN_INTERACTIONS = 10 
MAX_INTERACTIONS = 50 
MAX_SEQ_LEN = 30 
BATCH_SIZE = 64
PAD_ID = 0 
VOCAB_SIZE = NUM_ITEMS + 1

In [183]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Generate item categories


In [184]:
item_ids = np.arange(1,NUM_ITEMS+1)
item_categories = np.random.randint(0,NUM_CATEGORIES,size=NUM_ITEMS) 
items_df = pd.DataFrame({'item_id':item_ids,'category_id':item_categories}) 

In [185]:
items_df.head()

,item_id,category_id
0,1,6
1,2,3
2,3,7
3,4,4
4,5,6


## Generate User Item Interactions


In [186]:
interactions = []

for user_id in range(1,NUM_USERS+1): 
    favorite_categories = np.random.choice(NUM_CATEGORIES,size=2,replace=False)
    num_interactions = np.random.randint(MIN_INTERACTIONS,MAX_INTERACTIONS+1) 
    current_time = 0 

    for _ in range(num_interactions):
        use_favorite = np.random.rand() < 0.8
        if use_favorite:
            chosen_category = np.random.choice(favorite_categories)
        else: 
            chosen_category = np.random.randint(0,NUM_CATEGORIES) 
        candidate_items = items_df[items_df['category_id']==chosen_category]['item_id'].values

        chosen_item = np.random.choice(candidate_items) 

        interactions.append({
            "user_id":user_id,
            "item_id":int(chosen_item),
            "timestamp":current_time
        })

        current_time += 1
interactions_df = pd.DataFrame(interactions) 
interactions_df = interactions_df.sort_values(["user_id","timestamp"]).reset_index(drop=True) 

In [187]:
interactions_df.head()

,user_id,item_id,timestamp
0,1,347,0
1,1,77,1
2,1,206,2
3,1,460,3
4,1,240,4


## Turn interactions into sequences


In [188]:
user_sequences = interactions_df.groupby("user_id")['item_id'].apply(list).to_dict()

In [189]:
user_sequences[1]

[347,
 77,
 206,
 460,
 240,
 332,
 412,
 317,
 45,
 385,
 103,
 380,
 109,
 353,
 5,
 341,
 184,
 190,
 51,
 390,
 95,
 8,
 256,
 256,
 370,
 104,
 441,
 341]

## Train test split 


In [190]:
train_user_sequences = {}
test_user_sequences = []
for user_id, seq in user_sequences.items():
    if len(seq) < 2 : continue
    train_sequence = seq[:-1]
    target_item = seq[-1]
    train_user_sequences[user_id] = train_sequence
    test_user_sequences.append({
        "user_id": user_id,
        "input_sequence": train_sequence,
        "target_item": target_item
    })


In [191]:
train_user_sequences[1]

[347,
 77,
 206,
 460,
 240,
 332,
 412,
 317,
 45,
 385,
 103,
 380,
 109,
 353,
 5,
 341,
 184,
 190,
 51,
 390,
 95,
 8,
 256,
 256,
 370,
 104,
 441]

## Create Training Samples FOR SASRec


In [192]:
training_samples = []

for user_id,seq in train_user_sequences.items():
    for target in range(1,len(seq)): 
        input_seq = seq[:target] 
        target_item = seq[target]

        if len(input_seq) > MAX_SEQ_LEN:
            input_seq = input_seq[-MAX_SEQ_LEN:]

        training_samples.append({
            "user_id":user_id,
            "input_sequence":input_seq,
            "target_item":target_item
        })
samples_df = pd.DataFrame(training_samples)

In [193]:
training_samples[2]

{'user_id': 1, 'input_sequence': [347, 77, 206], 'target_item': 460}

In [194]:
samples_df.head()

,user_id,input_sequence,target_item
0,1,[347],77
1,1,"[347, 77]",206
2,1,"[347, 77, 206]",460
3,1,"[347, 77, 206, 460]",240
4,1,"[347, 77, 206, 460, 240]",332


## Create Custom Pytorch Dataset 

In [231]:
class SASRecDataset(Dataset):
    def __init__(self,samples,max_seq_len,pad_id = 0): 
        self.samples = samples
        self.max_seq_len = max_seq_len
        self.pad_id = pad_id
    def __len__(self):
        return len(self.samples)
    def __getitem__(self,idx):
        sample = self.samples[idx] 
        input_seq = sample["input_sequence"]
        target_item = sample["target_item"]

        padding_length = self.max_seq_len - len(input_seq)
        padded_sequence = [self.pad_id] * padding_length + input_seq

        input_tensor = torch.tensor(padded_sequence,dtype=torch.long)
        target_tensor = torch.tensor(target_item,dtype=torch.long)

        return input_tensor,target_tensor
train_dataset = SASRecDataset(training_samples,MAX_SEQ_LEN,PAD_ID)

In [196]:
input,target = train_dataset[0]
input,target

(tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0, 347]),
 tensor(77))

## Create Dataloader

In [197]:
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [198]:
input_batch,target_batch = next(iter(train_loader))
print("Input batch shape:",input_batch.shape)
print("Target batch shape:",target_batch.shape)
print("Input first item in batch example:\n",input_batch[0])
print("Target first item in batch example:\n",target_batch[0])

Input batch shape: torch.Size([64, 30])
Target batch shape: torch.Size([64])
Input first item in batch example:
 tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 242, 156, 347,
        388, 208])
Target first item in batch example:
 tensor(299)


## Create causal attention mask function 

In [199]:
def create_causal_mask(seq_len,device): 
    mask = torch. triu(
        torch.ones(seq_len,seq_len,device = device),
        diagonal = 1
    )
    mask = mask.masked_fill(mask==1,float("-inf"))
    mask = mask.masked_fill(mask==0,float(0.0))
    return mask

In [200]:
create_causal_mask(3,"cpu")

tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])

## Create Class SASRecModel

In [ ]:
class SASRecModel(nn.Module):
    def __init__(
        self,
        num_items,
        max_seq_len,
        embedding_dim=64,
        num_attention_heads=2,
        num_transformer_blocks=2,
        dropout_rate=0.1 
    ):
        super().__init__()
        self.num_items = num_items
        self.max_seq_len = max_seq_len
        self.embedding_dim = embedding_dim
        # 1. Item Embedding Layer
        self.item_embedding = nn.Embedding(
            num_embeddings = VOCAB_SIZE, 
            embedding_dim = embedding_dim, 
            padding_idx = 0 # Embedding for padding will be a zero vector and not updated during training
        )
        # 2. Position Embedding Layer
        self.position_embedding = nn.Embedding(
            num_embeddings = max_seq_len,
            embedding_dim = embedding_dim
        )

        self.dropout = nn.Dropout(dropout_rate) # Dropout layer for regularization

        # Transformer Blocks
        transformer_block = nn.TransformerEncoderLayer(
            # Multi-Head Self-Attention
            d_model = embedding_dim,  # Embedding dimension for the transformer, define the dim for matrices Q, K, V
            nhead = num_attention_heads, # Number of attention heads in the multi-head attention mechanism

            # Multi Layer Perceptron (MLP)
            dropout = dropout_rate, # Dropout rate for regularization
            dim_feedforward = embedding_dim * 4, # Expansion Weights 
            activation = "gelu", # Activation function for the feedforward network

            #Supplementary Arguments
            batch_first = True, # Input shape (batch_size, seq_len, embedding_dim)
            norm_first = True # Apply layer normalization before the attention and feedforward layers
        )
        # 3. Transformer Layer
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer = transformer_block, # Base transformer block to be repeated
            num_layers = num_transformer_blocks # Number of transformer blocks
        )

        self.layer_norm = nn.LayerNorm(embedding_dim) # Layer normalization to stabilize training

        # 4. Output Layer
        self.output_layer = nn.Linear(
            embedding_dim, # Input dimension from the transformer output
            VOCAB_SIZE # Output dimension equal to vocabulary size
        )
    def forward(self,input_sequences):
        device = input_sequences.device
        batch_size,seq_len = input_sequences.size() 

        positions = torch.arange(seq_len,device=device)
        positions = positions.unsqueeze(0) 
        positions = positions.expand(batch_size,seq_len) 

        item_emb = self.item_embedding(input_sequences) 
        pos_emb = self.position_embedding(positions) 
        
        x = item_emb + pos_emb 
        x = self.dropout(x) 

        causal_mask = create_causal_mask(seq_len,device) 
        padding_mask = input_sequences.eq(PAD_ID)

        x = self.transformer_encoder(
            x,
            mask = causal_mask,
            src_key_padding_mask = padding_mask
        )
        x = self.layer_norm(x) 

        # Extract the last hidden state corresponding to the last item in the sequence
        last_hidden_embeddings = x[:,-1,:] 

        # Using the last hidden state to predict the next item in the sequence
        logits = self.output_layer(last_hidden_embeddings) 

        return logits 


## II. CONSTANT VARIABLES FOR TRAINING

In [202]:
EMBEDDING_DIM = 256
NUM_ATTENTION_HEADS = 4
NUM_TRANSFORMER_BLOCKS = 4
DROPOUT_RATE = 0.2
NUM_EPOCHS = 50

## Initialize SASRec Model


In [203]:
model = SASRecModel(
    num_items = NUM_ITEMS, 
    max_seq_len = MAX_SEQ_LEN, 
    embedding_dim = EMBEDDING_DIM, 
    num_attention_heads = NUM_ATTENTION_HEADS, 
    num_transformer_blocks = NUM_TRANSFORMER_BLOCKS, 
    dropout_rate = DROPOUT_RATE 
)

/var/folders/v1/f1gz50591kl2_tm8_gxbrtcc0000gn/T/ipykernel_4157/4247958037.py:45: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(


## Testing Model with 1 batch

In [204]:
input_batch,target_batch = next(iter(train_loader))

In [205]:
input_batch,target_batch 

(tensor([[  0,   0,   0,  ..., 111,  73, 321],
         [  0,   0,   0,  ...,   0,   0, 235],
         [  0,   0,   0,  ..., 471, 211, 461],
         ...,
         [  0,   0,   0,  ..., 456,  18, 277],
         [  0,   0,   0,  ..., 244,  38, 228],
         [  0,   0,   0,  ..., 176, 211, 112]]),
 tensor([195, 257, 461, 253, 426, 179, 424,   6, 172,  99,  47, 352, 298, 188,
          92, 312,  96, 152, 298,  59, 418, 252, 120, 159, 319,  73, 197, 409,
         391,  48, 208, 182, 401, 220, 319, 377, 430,  53,  80, 370, 430, 157,
         317, 112, 451, 136,  68, 297, 311,  50, 144, 143, 495, 389,  17, 178,
          89, 101, 413,  82, 221, 382, 334, 256]))

In [206]:
input_batch.shape,target_batch.shape

(torch.Size([64, 30]), torch.Size([64]))

In [207]:
logits = model(input_batch)
print("Logits shape:",logits.shape)
print("Logits example:\n",logits)

Logits shape: torch.Size([64, 501])
Logits example:
 tensor([[-0.0139, -0.4687, -0.0574,  ...,  0.6020, -0.6001,  0.2788],
        [ 0.2243, -0.2272, -0.0831,  ..., -0.5912, -0.5532,  0.4505],
        [ 0.9614,  0.3346, -0.1464,  ...,  0.2621,  0.3230,  0.3914],
        ...,
        [ 0.4799, -0.3753, -0.5892,  ...,  0.2622, -0.1102,  0.7838],
        [-0.3363, -0.1743, -0.2380,  ..., -0.1404, -0.2361,  0.0915],
        [ 0.5890, -0.2553,  0.1636,  ...,  0.6850, -0.7746, -0.8349]],
       grad_fn=<AddmmBackward0>)


/Users/tranmy/WebApplication/LARAVEL/ml-recommendation-service/.venv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


In [208]:
loss_function = nn.CrossEntropyLoss(ignore_index=0) 

# Logits shape: (batch_size, num_items+1), Target shape: (batch_size,)
loss = loss_function(logits,target_batch) 
print("Loss:",loss.item())

Loss: 6.367631912231445


## Create Training Loop for SASRec

In [209]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") 
model = model.to(device)

In [210]:
loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001,
    weight_decay = 1e-6
)


In [211]:
for epoch in range(1,NUM_EPOCHS+1):
    model.train() 
    total_loss = 0.0 
    total_batches = 0 
    
    for input_batch,target_batch in train_loader:
        input_batch = input_batch.to(device)
        target_batch = target_batch.to(device) 
        # -------- Training Step --------
        # 1. Feed Forward
        logits = model(input_batch)

        # 2. Compute Loss
        loss = loss_function(logits,target_batch) 
        
        # 3. Reset gradients
        optimizer.zero_grad()

        # 4. Feed Backward
        loss.backward() 

        # 5. Update Parameters 
        optimizer.step()
        # -------- Training Step --------

        # Accumulate loss for monitoring 
        total_loss += loss.item()
        total_batches += 1
        
    avg_loss = total_loss / total_batches
    print(f"Epoch {epoch}/{NUM_EPOCHS}, Average Loss: {avg_loss:.4f}")

Epoch 1/50, Average Loss: 6.3076
Epoch 2/50, Average Loss: 6.1270
Epoch 3/50, Average Loss: 5.9461
Epoch 4/50, Average Loss: 5.8577
Epoch 5/50, Average Loss: 5.8077
Epoch 6/50, Average Loss: 5.7778
Epoch 7/50, Average Loss: 5.7451
Epoch 8/50, Average Loss: 5.7226
Epoch 9/50, Average Loss: 5.6905
Epoch 10/50, Average Loss: 5.6760
Epoch 11/50, Average Loss: 5.6546
Epoch 12/50, Average Loss: 5.6287
Epoch 13/50, Average Loss: 5.6082
Epoch 14/50, Average Loss: 5.6043
Epoch 15/50, Average Loss: 5.5895
Epoch 16/50, Average Loss: 5.5808
Epoch 17/50, Average Loss: 5.5886
Epoch 18/50, Average Loss: 5.5861
Epoch 19/50, Average Loss: 5.5889
Epoch 20/50, Average Loss: 5.5720
Epoch 21/50, Average Loss: 5.5595
Epoch 22/50, Average Loss: 5.5576
Epoch 23/50, Average Loss: 5.5568
Epoch 24/50, Average Loss: 5.6001
Epoch 25/50, Average Loss: 5.5668
Epoch 26/50, Average Loss: 5.5432
Epoch 27/50, Average Loss: 5.5364
Epoch 28/50, Average Loss: 5.5210
Epoch 29/50, Average Loss: 5.5043
Epoch 30/50, Average Lo

## Recommend for next items  

In [255]:
TOP_K = 50

In [256]:
def recommend(model,user_sequence,top_k=10): 
    model.eval()
    if len(user_sequence) > MAX_SEQ_LEN: 
        user_sequence = user_sequence[-MAX_SEQ_LEN:] 
    padding_length = MAX_SEQ_LEN - len(user_sequence)
    padded_sequence = [0] * padding_length + user_sequence

    input_tensor = torch.tensor(
        [padded_sequence], 
        dtype = torch.long
    ).to(device) 

    with torch.no_grad():
        logits = model(input_tensor) 
        scores = logits[0] 
    
    scores[0] = float("-inf")

    for item_id in set(user_sequence): 
        scores[item_id] = float("-inf")
    top_scores,top_items = torch.topk(scores,k=top_k)

    return top_items.cpu().tolist(),top_scores.cpu().tolist() 

In [257]:
example_user_id = 2
example_sequence = user_sequences[example_user_id]
recommended_items,recommended_scores = recommend(
    model=model,
    user_sequence=example_sequence,
    top_k=TOP_K
)
print("Past interacted items:\n", example_sequence)
print("Recommended items:\n ", recommended_items)
print("Recommended scores:\n ", recommended_scores)

Past interacted items:
 [328, 162, 376, 272, 24, 97, 124, 248, 218, 461, 376, 108, 162, 225, 50, 479, 355, 387, 274, 297, 213, 302, 3, 50, 272]
Recommended items:
  [9, 251, 493, 339, 68, 485, 154, 55, 346, 85, 400, 220, 217, 71, 318, 193, 13, 160, 86, 273, 455, 319, 100, 382, 196, 464, 18, 476, 308, 244, 12, 286, 105, 425, 89, 327, 437, 175, 183, 156, 75, 270, 83, 246, 416, 92, 445, 72, 423, 238]
Recommended scores:
  [3.303262233734131, 2.9617083072662354, 2.823052406311035, 2.626849412918091, 2.595945358276367, 2.5514907836914062, 2.4931578636169434, 2.4921982288360596, 2.414726495742798, 2.339686393737793, 2.2780673503875732, 2.2290894985198975, 2.2176504135131836, 2.2162153720855713, 2.140320301055908, 2.113978862762451, 2.108170747756958, 2.1076996326446533, 1.9946160316467285, 1.9703278541564941, 1.9554582834243774, 1.9023033380508423, 1.89988112449646, 1.8589086532592773, 1.8543264865875244, 1.852559208869934, 1.8503191471099854, 1.8228864669799805, 1.8118840456008911, 1.762329

In [258]:
category_from_recommended = items_df[items_df['item_id'].isin(recommended_items)]['category_id'].unique()
category_from_past = items_df[items_df['item_id'].isin(example_sequence)]['category_id'].unique()
print("Categories of recommended items:\n", category_from_recommended)
print("Categories of past interacted items:\n", category_from_past)

Categories of recommended items:
 [7 4 0 8 9]
Categories of past interacted items:
 [7 5 8 6 4 9]


## Evaluation

In [261]:
def evaluate_model(model,test_samples,top_k=10): 
    hits = []
    recalls = []
    ndcgs = []
    for sample in test_samples:
        user_sequence = sample["input_sequence"]
        target_item = sample["target_item"]
        recommended_items,recommended_scores = recommend(
            model=model,
            user_sequence=user_sequence,
            top_k=top_k
        )

        if target_item in recommended_items: 
            hit = 1
            recall = 1 
            rank = recommended_items.index(target_item) + 1 
            ndcg = 1/math.log2(rank+1) 
        else:
            hit = 0 
            recall = 0 
            ndcg = 0
        hits.append(hit)
        recalls.append(recall)
        ndcgs.append(ndcg)  
    hit_at_k = sum(hits)/len(hits)
    recall_at_k = sum(recalls)/len(recalls)
    ndcg_at_k = sum(ndcgs)/len(ndcgs)

    return hit_at_k, recall_at_k, ndcg_at_k, hits, recalls, ndcgs

In [262]:
hit_at_10, recall_at_10, ndcg_at_10 ,hits, recalls, ndcgs = evaluate_model(model,test_user_sequences,top_k=10)
print(f"Hit@10: {hit_at_10:.4f}, Recall@10: {recall_at_10:.4f}, NDCG@10: {ndcg_at_10:.4f}") 

/Users/tranmy/WebApplication/LARAVEL/ml-recommendation-service/.venv/lib/python3.14/site-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


Hit@10: 0.0740, Recall@10: 0.0740, NDCG@10: 0.0341


In [263]:
hits.count(1)

74

In [264]:
len(test_user_sequences)

1000